In [5]:
import pandas as pd
import numpy as np


In [6]:

def load_and_process_data(csv_path, condition_label):
    """Load CSV and add condition column."""
    df = pd.read_csv(csv_path)
    df['condition'] = condition_label
    return df

def summarize_interface_stats(df, interface_name):
    """Summarize statistics for a specific interface across conditions."""
    interface_df = df[df['column'] == interface_name].copy()
    
    summary = []
    for condition in interface_df['condition'].unique():
        cond_df = interface_df[interface_df['condition'] == condition]
        
        # Calculate statistics
        mean_diff = cond_df['mean_diff_series_minus_ref'].mean()
        cohen_d = cond_df['cohen_d_paired'].mean()
        n_significant = cond_df['lower_by_both'].sum()
        n_total = len(cond_df)
        
        # Get min p-value for significant results
        sig_pvals = cond_df[cond_df['lower_by_both']]['p_perm_one_sided_lower_fdr']
        p_val_str = f"<0.001" if len(sig_pvals) > 0 and sig_pvals.min() < 0.001 else "NS"
        
        summary.append({
            'Interface': interface_name,
            'Salt Condition': condition,
            'Mean Difference (kJ/mol)': f"{mean_diff:.2f}",
            "Cohen's d": f"{cohen_d:.2f}",
            'p-value (FDR)': p_val_str,
            'Haplotypes Significantly Lower': f"{n_significant}/{n_total}"
        })
    
    return summary

def create_table1_hap6():
    """Table 1: HAP6 vs Reference across all conditions."""
    
    # Load all HAP6 comparison files
    df_1M = load_and_process_data(
        '../data/interim/results_statistics/energy_compare_hap6_vs_special_key_hap7.csv',
        '1M NaCl'
    )
    df_14mM = load_and_process_data(
        '../data/interim/results_statistics/energy_compare_hap6_vs_special_key_hap7_14mM_NaCl.csv',
        '14mM NaCl'
    )
    df_5mM_Ca = load_and_process_data(
        '../data/interim/results_statistics/energy_compare_hap6_vs_special_key_hap7_5mM_Ca2.csv',
        '5mM Ca²⁺'
    )
    
    # Combine all conditions
    df_all = pd.concat([df_1M, df_14mM, df_5mM_Ca], ignore_index=True)
    
    # Focus on the 4 main interfaces
    interfaces = [
        'Coul-SR:interface_CIPK17_to_CBL8-interface_CBL8_to_CIPK17',
        'LJ-SR:interface_CIPK17_to_CBL8-interface_CBL8_to_CIPK17',
        'Coul-SR:interface_NAC77_to_CIPK17-interface_CIPK17_to_NAC77',
        'LJ-SR:interface_NAC77_to_CIPK17-interface_CIPK17_to_NAC77'
    ]
    
    # Generate summary for each interface
    all_summaries = []
    for interface in interfaces:
        all_summaries.extend(summarize_interface_stats(df_all, interface))
    
    # Create DataFrame
    table1 = pd.DataFrame(all_summaries)
    
    # Simplify interface names for display
    table1['Interface'] = table1['Interface'].replace({
        'Coul-SR:interface_CIPK17_to_CBL8-interface_CBL8_to_CIPK17': '**Coul-SR: CIPK17↔CBL8**',
        'LJ-SR:interface_CIPK17_to_CBL8-interface_CBL8_to_CIPK17': '**LJ-SR: CIPK17↔CBL8**',
        'Coul-SR:interface_NAC77_to_CIPK17-interface_CIPK17_to_NAC77': '**Coul-SR: NAC77↔CIPK17**',
        'LJ-SR:interface_NAC77_to_CIPK17-interface_CIPK17_to_NAC77': '**LJ-SR: NAC77↔CIPK17**'
    })
    
    return table1

def create_table2_hap7():
    """Table 2: HAP7 vs Reference across all conditions."""
    
    # Load all HAP7 comparison files
    df_1M = load_and_process_data(
        '../data/interim/results_statistics/energy_compare_hap7_vs_special_key_hap7.csv',
        '1M NaCl'
    )
    df_14mM = load_and_process_data(
        '../data/interim/results_statistics/energy_compare_hap7_vs_special_key_hap7_14mM_NaCl.csv',
        '14mM NaCl'
    )
    df_5mM_Ca = load_and_process_data(
        '../data/interim/results_statistics/energy_compare_hap7_vs_special_key_hap7_5mM_Ca2.csv',
        '5mM Ca²⁺'
    )
    
    # Combine all conditions
    df_all = pd.concat([df_1M, df_14mM, df_5mM_Ca], ignore_index=True)
    
    # Focus on the 4 main interfaces
    interfaces = [
        'Coul-SR:interface_CIPK17_to_CBL8-interface_CBL8_to_CIPK17',
        'LJ-SR:interface_CIPK17_to_CBL8-interface_CBL8_to_CIPK17',
        'Coul-SR:interface_NAC77_to_CIPK17-interface_CIPK17_to_NAC77',
        'LJ-SR:interface_NAC77_to_CIPK17-interface_CIPK17_to_NAC77'
    ]
    
    # Generate summary for each interface
    all_summaries = []
    for interface in interfaces:
        all_summaries.extend(summarize_interface_stats(df_all, interface))
    
    # Create DataFrame
    table2 = pd.DataFrame(all_summaries)
    
    # Simplify interface names
    table2['Interface'] = table2['Interface'].replace({
        'Coul-SR:interface_CIPK17_to_CBL8-interface_CBL8_to_CIPK17': '**Coul-SR: CIPK17↔CBL8**',
        'LJ-SR:interface_CIPK17_to_CBL8-interface_CBL8_to_CIPK17': '**LJ-SR: CIPK17↔CBL8**',
        'Coul-SR:interface_NAC77_to_CIPK17-interface_CIPK17_to_NAC77': '**Coul-SR: NAC77↔CIPK17**',
        'LJ-SR:interface_NAC77_to_CIPK17-interface_CIPK17_to_NAC77': '**LJ-SR: NAC77↔CIPK17**'
    })
    
    return table2

def create_table3_nac77_detailed():
    """Table 3: Detailed NAC77-CIPK17 interface analysis."""
    
    # Load all files
    hap6_1M = load_and_process_data(
        '../data/interim/results_statistics/energy_compare_hap6_vs_special_key_hap7.csv',
        '1M NaCl'
    )
    hap6_14mM = load_and_process_data(
        '../data/interim/results_statistics/energy_compare_hap6_vs_special_key_hap7_14mM_NaCl.csv',
        '14mM NaCl'
    )
    hap6_5mM = load_and_process_data(
        '../data/interim/results_statistics/energy_compare_hap6_vs_special_key_hap7_5mM_Ca2.csv',
        '5mM Ca²⁺'
    )
    
    hap7_1M = load_and_process_data(
        '../data/interim/results_statistics/energy_compare_hap7_vs_special_key_hap7.csv',
        '1M NaCl'
    )
    hap7_14mM = load_and_process_data(
        '../data/interim/results_statistics/energy_compare_hap7_vs_special_key_hap7_14mM_NaCl.csv',
        '14mM NaCl'
    )
    hap7_5mM = load_and_process_data(
        '../data/interim/results_statistics/energy_compare_hap7_vs_special_key_hap7_5mM_Ca2.csv',
        '5mM Ca²⁺'
    )
    
    # Combine
    hap6_all = pd.concat([hap6_1M, hap6_14mM, hap6_5mM], ignore_index=True)
    hap7_all = pd.concat([hap7_1M, hap7_14mM, hap7_5mM], ignore_index=True)
    
    # NAC77 interfaces only
    nac77_interfaces = [
        # 'Coul-SR:interface_NAC77_to_CIPK17-interface_NAC77_to_CIPK17',
        # 'LJ-SR:interface_NAC77_to_CIPK17-interface_NAC77_to_CIPK17',
        'Coul-SR:interface_NAC77_to_CIPK17-interface_CIPK17_to_NAC77',
        'LJ-SR:interface_NAC77_to_CIPK17-interface_CIPK17_to_NAC77'
    ]
    
    summary = []
    for interface in nac77_interfaces:
        for condition in ['1M NaCl', '14mM NaCl', '5mM Ca²⁺']:
            # HAP6 stats
            hap6_df = hap6_all[(hap6_all['column'] == interface) & (hap6_all['condition'] == condition)]
            hap6_mean_range = f"{hap6_df['mean_diff_series_minus_ref'].min():.2f} to {hap6_df['mean_diff_series_minus_ref'].max():.2f}"
            hap6_cohen_range = f"{hap6_df['cohen_d_paired'].min():.2f} to {hap6_df['cohen_d_paired'].max():.2f}"
            hap6_n_sig = f"{hap6_df['lower_by_both'].sum()}/{len(hap6_df)}"
            
            # HAP7 stats
            hap7_df = hap7_all[(hap7_all['column'] == interface) & (hap7_all['condition'] == condition)]
            hap7_n_sig = f"{hap7_df['lower_by_both'].sum()}/{len(hap7_df)}"
            
            # Determine effect direction
            mean_val = hap6_df['mean_diff_series_minus_ref'].mean()
            if mean_val < -10:
                direction = "Destabilizing"
            elif mean_val > 10:
                direction = "Stabilizing"
            else:
                direction = "Mixed"
            
            summary.append({
                'Interface': interface,
                'Salt Condition': condition,
                'Mean Difference (kJ/mol)': hap6_mean_range,
                "Cohen's d": hap6_cohen_range,
                'Effect Direction': direction,
                'n Significant (HAP6/HAP7)': f"{hap6_n_sig}; {hap7_n_sig}"
            })
    
    table3 = pd.DataFrame(summary)
    
    # Simplify interface names
    table3['Interface'] = table3['Interface'].replace({
        # 'Coul-SR:interface_NAC77_to_CIPK17-interface_NAC77_to_CIPK17': '**Coul-SR: NAC77↔CIPK17**',
        # 'LJ-SR:interface_NAC77_to_CIPK17-interface_NAC77_to_CIPK17': '**LJ-SR: NAC77↔CIPK17**',
        'Coul-SR:interface_NAC77_to_CIPK17-interface_CIPK17_to_NAC77': '**Coul-SR: NAC77-CIPK17**',
        'LJ-SR:interface_NAC77_to_CIPK17-interface_CIPK17_to_NAC77': '**LJ-SR: NAC77-CIPK17**'
    })
    
    return table3

def find_haplotypes_lower_in_both_energies():
    """Find haplotypes significantly lower in both Coul-SR AND LJ-SR for each interface."""
    
    # Load all files
    files = {
        'HAP6_1M': '../data/interim/results_statistics/energy_compare_hap6_vs_special_key_hap7.csv',
        'HAP6_14mM': '../data/interim/results_statistics/energy_compare_hap6_vs_special_key_hap7_14mM_NaCl.csv',
        'HAP6_5mM_Ca': '../data/interim/results_statistics/energy_compare_hap6_vs_special_key_hap7_5mM_Ca2.csv',
        'HAP7_1M': '../data/interim/results_statistics/energy_compare_hap7_vs_special_key_hap7.csv',
        'HAP7_14mM': '../data/interim/results_statistics/energy_compare_hap7_vs_special_key_hap7_14mM_NaCl.csv',
        'HAP7_5mM_Ca': '../data/interim/results_statistics/energy_compare_hap7_vs_special_key_hap7_5mM_Ca2.csv',
    }
    
    results = []
    
    for label, filepath in files.items():
        df = pd.read_csv(filepath)
        
        # Define interface pairs (Coul-SR and LJ-SR for same interface)
        interface_pairs = [
            ('Coul-SR:interface_CIPK17_to_CBL8-interface_CBL8_to_CIPK17',
             'LJ-SR:interface_CIPK17_to_CBL8-interface_CBL8_to_CIPK17',
             'CIPK17↔CBL8'),
            ('Coul-SR:interface_NAC77_to_CIPK17-interface_CIPK17_to_NAC77',
             'LJ-SR:interface_NAC77_to_CIPK17-interface_CIPK17_to_NAC77',
             'NAC77↔CIPK17')
        ]
        
        for coul_interface, lj_interface, interface_name in interface_pairs:
            # Get haplotypes significant in Coul-SR
            coul_sig = df[(df['column'] == coul_interface) & 
                         (df['lower_by_both'] == True)]['series_key'].tolist()
            
            # Get haplotypes significant in LJ-SR
            lj_sig = df[(df['column'] == lj_interface) & 
                       (df['lower_by_both'] == True)]['series_key'].tolist()
            
            # Find intersection (haplotypes significant in BOTH)
            both_sig = list(set(coul_sig) & set(lj_sig))
            
            # Get statistics for these haplotypes
            for haplotype in both_sig:
                coul_row = df[(df['column'] == coul_interface) & 
                             (df['series_key'] == haplotype)].iloc[0]
                lj_row = df[(df['column'] == lj_interface) & 
                           (df['series_key'] == haplotype)].iloc[0]
                
                results.append({
                    'Dataset': label,
                    'Interface': interface_name,
                    'Haplotype': haplotype,
                    'Coul-SR Mean Diff': f"{coul_row['mean_diff_series_minus_ref']:.2f}",
                    'Coul-SR Cohen d': f"{coul_row['cohen_d_paired']:.2f}",
                    'Coul-SR p-value': f"{coul_row['p_perm_one_sided_lower_fdr']:.4f}",
                    'LJ-SR Mean Diff': f"{lj_row['mean_diff_series_minus_ref']:.2f}",
                    'LJ-SR Cohen d': f"{lj_row['cohen_d_paired']:.2f}",
                    'LJ-SR p-value': f"{lj_row['p_perm_one_sided_lower_fdr']:.4f}",
                })
    
    return pd.DataFrame(results)


In [7]:

# Generate all tables
print("=" * 80)
print("TABLE 1: HAP6 vs Reference (HAP7-HAP56)")
print("=" * 80)
table1 = create_table1_hap6()
print(table1.to_markdown(index=False))
print("\n")

print("=" * 80)
print("TABLE 2: HAP7 vs Reference (HAP7-HAP56)")
print("=" * 80)
table2 = create_table2_hap7()
print(table2.to_markdown(index=False))
print("\n")

print("=" * 80)
print("TABLE 3: NAC77-CIPK17 Interface (Ionic Condition Effects)")
print("=" * 80)
table3 = create_table3_nac77_detailed()
print(table3.to_markdown(index=False))
print("\n")

# Save to CSV
table1.to_csv('../data/interim/results_statistics/publication_table1_hap6.csv', index=False)
table2.to_csv('../data/interim/results_statistics/publication_table2_hap7.csv', index=False)
table3.to_csv('../data/interim/results_statistics/publication_table3_nac77.csv', index=False)

print("✓ Tables saved to ../data/interim/results_statistics/")

# Generate the table
print("=" * 100)
print("HAPLOTYPES SIGNIFICANTLY LOWER IN BOTH COUL-SR AND LJ-SR")
print("=" * 100)
both_lower_table = find_haplotypes_lower_in_both_energies()
print(both_lower_table.to_markdown(index=False))

# Save to CSV
both_lower_table.to_csv('../data/interim/results_statistics/haplotypes_lower_both_energies.csv', index=False)
print("\n✓ Saved to ../data/interim/results_statistics/haplotypes_lower_both_energies.csv")

# Summary statistics
print("\n" + "=" * 100)
print("SUMMARY: Count of haplotypes lower in BOTH energies per dataset/interface")
print("=" * 100)
summary = both_lower_table.groupby(['Dataset', 'Interface']).size().reset_index(name='Count')
print(summary.to_markdown(index=False))

TABLE 1: HAP6 vs Reference (HAP7-HAP56)
| Interface                 | Salt Condition   |   Mean Difference (kJ/mol) |   Cohen's d | p-value (FDR)   | Haplotypes Significantly Lower   |
|:--------------------------|:-----------------|---------------------------:|------------:|:----------------|:---------------------------------|
| **Coul-SR: CIPK17↔CBL8**  | 1M NaCl          |                     -10.15 |       -0.11 | <0.001          | 3/6                              |
| **Coul-SR: CIPK17↔CBL8**  | 14mM NaCl        |                     -14.69 |       -0.14 | <0.001          | 5/6                              |
| **Coul-SR: CIPK17↔CBL8**  | 5mM Ca²⁺         |                     -42.24 |       -0.42 | <0.001          | 5/6                              |
| **LJ-SR: CIPK17↔CBL8**    | 1M NaCl          |                     -19.32 |       -0.65 | <0.001          | 6/6                              |
| **LJ-SR: CIPK17↔CBL8**    | 14mM NaCl        |                     -18.8  |       -0.65 

In [8]:
def create_summary_table_both_lower():
    """Create a compact summary table showing counts and key statistics."""
    
    both_df = find_haplotypes_lower_in_both_energies()
    
    summary = []
    for dataset in both_df['Dataset'].unique():
        for interface in both_df['Interface'].unique():
            subset = both_df[(both_df['Dataset'] == dataset) & 
                           (both_df['Interface'] == interface)]
            
            if len(subset) > 0:
                # Extract numeric values for averaging
                coul_means = subset['Coul-SR Mean Diff'].str.replace(' ', '').astype(float)
                lj_means = subset['LJ-SR Mean Diff'].str.replace(' ', '').astype(float)
                coul_cohens = subset['Coul-SR Cohen d'].str.replace(' ', '').astype(float)
                lj_cohens = subset['LJ-SR Cohen d'].str.replace(' ', '').astype(float)
                
                summary.append({
                    'Dataset': dataset,
                    'Interface': interface,
                    'n Haplotypes': len(subset),
                    'Avg Coul-SR Diff': f"{coul_means.mean():.2f}",
                    'Avg LJ-SR Diff': f"{lj_means.mean():.2f}",
                    'Avg Coul-SR Cohen d': f"{coul_cohens.mean():.2f}",
                    'Avg LJ-SR Cohen d': f"{lj_cohens.mean():.2f}",
                    'Haplotypes': ', '.join([h.split('_CIPK17_')[1].split('_NAC77')[0] 
                                            for h in subset['Haplotype'].tolist()])
                })
    
    return pd.DataFrame(summary)

print("\n" + "=" * 100)
print("COMPACT SUMMARY TABLE")
print("=" * 100)
summary_table = create_summary_table_both_lower()
print(summary_table.to_markdown(index=False))

# Save
summary_table.to_csv('../data/interim/results_statistics/summary_both_energies_lower.csv', index=False)
print("\n✓ Saved to ../data/interim/results_statistics/summary_both_energies_lower.csv")


COMPACT SUMMARY TABLE
| Dataset     | Interface    |   n Haplotypes |   Avg Coul-SR Diff |   Avg LJ-SR Diff |   Avg Coul-SR Cohen d |   Avg LJ-SR Cohen d | Haplotypes                                                        |
|:------------|:-------------|---------------:|-------------------:|-----------------:|----------------------:|--------------------:|:------------------------------------------------------------------|
| HAP6_1M     | CIPK17↔CBL8  |              3 |             -71.68 |           -13.34 |                 -0.82 |               -0.47 | HAP56_Freq.6124, HAP55_Freq.4952, HAP54_Freq.2103                 |
| HAP6_14mM   | CIPK17↔CBL8  |              4 |             -34.57 |           -26.27 |                 -0.34 |               -0.9  | HAP53_Freq.1974, HAP51_Freq.492, HAP56_Freq.6124, HAP55_Freq.4952 |
| HAP6_5mM_Ca | CIPK17↔CBL8  |              2 |             -46.43 |           -19.7  |                 -0.52 |               -0.66 | HAP51_Freq.492, HAP55_Freq.4952    

In [9]:
def create_coulsr_focused_table():
    """Create comprehensive table for Coul-SR interactions only."""
    
    # Load all files
    files = {
        'HAP6_1M': '../data/interim/results_statistics/energy_compare_hap6_vs_special_key_hap7.csv',
        'HAP6_14mM': '../data/interim/results_statistics/energy_compare_hap6_vs_special_key_hap7_14mM_NaCl.csv',
        'HAP6_5mM_Ca': '../data/interim/results_statistics/energy_compare_hap6_vs_special_key_hap7_5mM_Ca2.csv',
        'HAP7_1M': '../data/interim/results_statistics/energy_compare_hap7_vs_special_key_hap7.csv',
        'HAP7_14mM': '../data/interim/results_statistics/energy_compare_hap7_vs_special_key_hap7_14mM_NaCl.csv',
        'HAP7_5mM_Ca': '../data/interim/results_statistics/energy_compare_hap7_vs_special_key_hap7_5mM_Ca2.csv',
    }
    
    # Coul-SR interfaces only
    coulsr_interfaces = [
        'Coul-SR:interface_CIPK17_to_CBL8-interface_CBL8_to_CIPK17',
        'Coul-SR:interface_NAC77_to_CIPK17-interface_CIPK17_to_NAC77'
    ]
    
    results = []
    
    for label, filepath in files.items():
        df = pd.read_csv(filepath)
        
        for interface in coulsr_interfaces:
            interface_df = df[df['column'] == interface]
            
            # Get significant haplotypes
            sig_df = interface_df[interface_df['lower_by_both'] == True]
            
            for _, row in sig_df.iterrows():
                results.append({
                    'Dataset': label,
                    'Interface': interface,
                    'Haplotype': row['series_key'],
                    'Mean Diff (kJ/mol)': f"{row['mean_diff_series_minus_ref']:.2f}",
                    "Cohen's d": f"{row['cohen_d_paired']:.2f}",
                    'p-value (FDR)': f"{row['p_perm_one_sided_lower_fdr']:.4f}",
                    't-statistic': f"{row['t_stat']:.2f}"
                })
    
    coulsr_table = pd.DataFrame(results)
    
    # Simplify interface names
    coulsr_table['Interface'] = coulsr_table['Interface'].replace({
        'Coul-SR:interface_CIPK17_to_CBL8-interface_CBL8_to_CIPK17': 'CIPK17↔CBL8',
        'Coul-SR:interface_NAC77_to_CIPK17-interface_CIPK17_to_NAC77': 'NAC77↔CIPK17'
    })
    
    return coulsr_table

def create_coulsr_summary_by_condition():
    """Summary table: Coul-SR grouped by dataset and interface."""
    
    coulsr_full = create_coulsr_focused_table()
    
    summary = []
    
    for dataset in coulsr_full['Dataset'].unique():
        for interface in coulsr_full['Interface'].unique():
            subset = coulsr_full[(coulsr_full['Dataset'] == dataset) & 
                                (coulsr_full['Interface'] == interface)]
            
            if len(subset) > 0:
                # Extract numeric values
                mean_diffs = subset['Mean Diff (kJ/mol)'].str.replace(' ', '').astype(float)
                cohens = subset["Cohen's d"].str.replace(' ', '').astype(float)
                
                summary.append({
                    'Dataset': dataset,
                    'Interface': interface,
                    'n Significant': len(subset),
                    'Mean Diff Range': f"{mean_diffs.min():.2f} to {mean_diffs.max():.2f}",
                    'Avg Mean Diff': f"{mean_diffs.mean():.2f}",
                    'Avg Cohen d': f"{cohens.mean():.2f}",
                    'Haplotypes': ', '.join([h.split('_CIPK17_')[1].split('_NAC77')[0] 
                                            for h in subset['Haplotype'].tolist()])
                })
    
    return pd.DataFrame(summary)

def create_coulsr_comparison_across_conditions():
    """Compare Coul-SR effects across the three salt conditions."""
    
    # Load files
    hap6_1M = pd.read_csv('../data/interim/results_statistics/energy_compare_hap6_vs_special_key_hap7.csv')
    hap6_14mM = pd.read_csv('../data/interim/results_statistics/energy_compare_hap6_vs_special_key_hap7_14mM_NaCl.csv')
    hap6_5mM = pd.read_csv('../data/interim/results_statistics/energy_compare_hap6_vs_special_key_hap7_5mM_Ca2.csv')
    
    hap7_1M = pd.read_csv('../data/interim/results_statistics/energy_compare_hap7_vs_special_key_hap7.csv')
    hap7_14mM = pd.read_csv('../data/interim/results_statistics/energy_compare_hap7_vs_special_key_hap7_14mM_NaCl.csv')
    hap7_5mM = pd.read_csv('../data/interim/results_statistics/energy_compare_hap7_vs_special_key_hap7_5mM_Ca2.csv')
    
    coulsr_interfaces = [
        'Coul-SR:interface_CIPK17_to_CBL8-interface_CBL8_to_CIPK17',
        'Coul-SR:interface_NAC77_to_CIPK17-interface_CIPK17_to_NAC77'
    ]
    
    comparison = []
    
    for interface in coulsr_interfaces:
        interface_name = 'CIPK17↔CBL8' if 'CBL8' in interface else 'NAC77↔CIPK17'
        
        # HAP6 analysis
        hap6_1M_data = hap6_1M[hap6_1M['column'] == interface]
        hap6_14mM_data = hap6_14mM[hap6_14mM['column'] == interface]
        hap6_5mM_data = hap6_5mM[hap6_5mM['column'] == interface]
        
        comparison.append({
            'Haplotype Set': 'HAP6',
            'Interface': interface_name,
            '1M NaCl - n sig': hap6_1M_data['lower_by_both'].sum(),
            '1M NaCl - Avg Diff': f"{hap6_1M_data[hap6_1M_data['lower_by_both']]['mean_diff_series_minus_ref'].mean():.2f}" if hap6_1M_data['lower_by_both'].sum() > 0 else "N/A",
            '14mM NaCl - n sig': hap6_14mM_data['lower_by_both'].sum(),
            '14mM NaCl - Avg Diff': f"{hap6_14mM_data[hap6_14mM_data['lower_by_both']]['mean_diff_series_minus_ref'].mean():.2f}" if hap6_14mM_data['lower_by_both'].sum() > 0 else "N/A",
            '5mM Ca²⁺ - n sig': hap6_5mM_data['lower_by_both'].sum(),
            '5mM Ca²⁺ - Avg Diff': f"{hap6_5mM_data[hap6_5mM_data['lower_by_both']]['mean_diff_series_minus_ref'].mean():.2f}" if hap6_5mM_data['lower_by_both'].sum() > 0 else "N/A"
        })
        
        # HAP7 analysis
        hap7_1M_data = hap7_1M[hap7_1M['column'] == interface]
        hap7_14mM_data = hap7_14mM[hap7_14mM['column'] == interface]
        hap7_5mM_data = hap7_5mM[hap7_5mM['column'] == interface]
        
        comparison.append({
            'Haplotype Set': 'HAP7',
            'Interface': interface_name,
            '1M NaCl - n sig': hap7_1M_data['lower_by_both'].sum(),
            '1M NaCl - Avg Diff': f"{hap7_1M_data[hap7_1M_data['lower_by_both']]['mean_diff_series_minus_ref'].mean():.2f}" if hap7_1M_data['lower_by_both'].sum() > 0 else "N/A",
            '14mM NaCl - n sig': hap7_14mM_data['lower_by_both'].sum(),
            '14mM NaCl - Avg Diff': f"{hap7_14mM_data[hap7_14mM_data['lower_by_both']]['mean_diff_series_minus_ref'].mean():.2f}" if hap7_14mM_data['lower_by_both'].sum() > 0 else "N/A",
            '5mM Ca²⁺ - n sig': hap7_5mM_data['lower_by_both'].sum(),
            '5mM Ca²⁺ - Avg Diff': f"{hap7_5mM_data[hap7_5mM_data['lower_by_both']]['mean_diff_series_minus_ref'].mean():.2f}" if hap7_5mM_data['lower_by_both'].sum() > 0 else "N/A"
        })
    
    return pd.DataFrame(comparison)

# Generate Coul-SR focused tables
print("=" * 100)
print("TABLE: ALL SIGNIFICANTLY LOWER COUL-SR INTERACTIONS")
print("=" * 100)
coulsr_full = create_coulsr_focused_table()
print(coulsr_full.to_markdown(index=False))

# Save
coulsr_full.to_csv('../data/interim/results_statistics/coulsr_all_significant.csv', index=False)
print("\n✓ Saved to ../data/interim/results_statistics/coulsr_all_significant.csv")

print("\n" + "=" * 100)
print("TABLE: COUL-SR SUMMARY BY DATASET AND INTERFACE")
print("=" * 100)
coulsr_summary = create_coulsr_summary_by_condition()
print(coulsr_summary.to_markdown(index=False))

# Save
coulsr_summary.to_csv('../data/interim/results_statistics/coulsr_summary_by_condition.csv', index=False)
print("\n✓ Saved to ../data/interim/results_statistics/coulsr_summary_by_condition.csv")

print("\n" + "=" * 100)
print("TABLE: COUL-SR COMPARISON ACROSS SALT CONDITIONS")
print("=" * 100)
coulsr_comparison = create_coulsr_comparison_across_conditions()
print(coulsr_comparison.to_markdown(index=False))

# Save
coulsr_comparison.to_csv('../data/interim/results_statistics/coulsr_comparison_across_conditions.csv', index=False)
print("\n✓ Saved to ../data/interim/results_statistics/coulsr_comparison_across_conditions.csv")

TABLE: ALL SIGNIFICANTLY LOWER COUL-SR INTERACTIONS
| Dataset     | Interface    | Haplotype                                                          |   Mean Diff (kJ/mol) |   Cohen's d |   p-value (FDR) |   t-statistic |
|:------------|:-------------|:-------------------------------------------------------------------|---------------------:|------------:|----------------:|--------------:|
| HAP6_1M     | CIPK17↔CBL8  | CBL8_HAP6_Freq.3062_CIPK17_HAP54_Freq.2103_NAC77_HAP21_Freq.13677  |              -140.2  |       -1.65 |          0.0005 |       -180.64 |
| HAP6_1M     | CIPK17↔CBL8  | CBL8_HAP6_Freq.3062_CIPK17_HAP55_Freq.4952_NAC77_HAP21_Freq.13677  |               -12.37 |       -0.14 |          0.0005 |        -15.75 |
| HAP6_1M     | CIPK17↔CBL8  | CBL8_HAP6_Freq.3062_CIPK17_HAP56_Freq.6124_NAC77_HAP21_Freq.13677  |               -62.48 |       -0.68 |          0.0005 |        -74.2  |
| HAP6_1M     | NAC77↔CIPK17 | CBL8_HAP6_Freq.3062_CIPK17_HAP54_Freq.2103_NAC77_HAP21_Freq.13